# Notebook 03 — Clinical associations

For manuscript parity, this notebook runs the Data S1 clinical tests across all 547 components before filtering to selected components: sex (Mann-Whitney U), CERAD 1/2 vs 3/4 (Mann-Whitney U), and cogdx groups 1/2/4 (Kruskal-Wallis). The displayed `p_adjusted` values are Benjamini-Hochberg adjusted separately for each clinical variable.

**Required input.** `ROSMAP_clinical.csv` — controlled access; request via the [AD Knowledge Portal DUC](https://adknowledgeportal.synapse.org/Data%20Access). Place it in `references/` next to the reference tables (or set `G_ROS_CLINICAL_PATH`) and update `CLINICAL_PATH` below if needed.

## Required Synapse files

Download version 3 of Synapse dataset `syn75275226` (AD Knowledge Portal). Place all files in a
single directory and set `DEPOSIT_DIR` below. Component IDs are 1–519 and 521–548
(ID 520 is retired), so select components by ID, not by row number.

| File | Shape | Description |
|---|---|---|
| `samples_final.csv` | 547 × 276 | Component × sample loadings |
| `cell_type.csv` | 547 × 6 | Component × cell-type loadings (Ast, Oli, Inh, OPC, Exc, Mic) |
| `RNA.txt` | 25,098 × 547 | Gene × component loadings (PIP > 0.5 masked) |
| `meth.txt` | 51,798 × 547 | CpG × component loadings (PIP > 0.5 masked) |
| `chip.txt` | 26,384 × 547 | H3K9ac peak × component loadings (PIP > 0.5 masked) |

## Required reference annotations

Not bundled with this repository: download the three tables into `references/` (sources and steps in `references/README.md`). The gene annotation is extracted from the Ensembl release 100 GTF; the array manifest and the peak coordinates come with the ROSMAP DNA methylation and H3K9ac ChIP-seq data on the AD Knowledge Portal. All four notebooks load them with the deposit.

| File | Description |
|---|---|
| `Homo_sapiens.GRCh38.100.txt` | Gene annotation (Ensembl 100) |
| `ROSMAP_arrayMethylation_metaData.tsv` | Illumina methylation array manifest |
| `H3K9acDomains.csv` | H3K9ac ChIP-seq peak coordinates (Klein et al.) |

Clinical metadata (`ROSMAP_clinical.csv`) is **not redistributed** — it is
controlled-access (request via the
[AD Knowledge Portal DUC](https://adknowledgeportal.synapse.org/Data%20Access)). Notebooks that need
clinical variables document their dependency inline and degrade gracefully
when the file is absent.


## Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the repo root to sys.path so we can import the helper module.
sys.path.insert(0, "..")
import g_ros_release as gr

gr.apply_plot_style()


In [2]:
# Path configuration — edit these two paths to match your local layout.
# `DEPOSIT_DIR` is where you downloaded the Synapse files (dataset
# syn75275226, version 3). `REFERENCES_DIR` is the `references/`
# directory in this repo, where you put the reference tables.

# DEPOSIT_DIR is auto-resolved between two common layouts:
#   1. `synapse/` inside the cloned repo (default — what end-users get when
#      they download the deposit into a `synapse/` folder at the repo root)
#   2. `synapse/` as a sibling of the repo (the local-prep layout used to
#      keep code and data physically separate during preparation)
# To override either default, set the env var G_ROS_DEPOSIT_DIR.
import os as _os
DEPOSIT_DIR = _os.environ.get("G_ROS_DEPOSIT_DIR", "../synapse")
if not _os.path.exists(_os.path.join(DEPOSIT_DIR, "samples_final.csv")):
    _fallback = "../../synapse"
    if _os.path.exists(_os.path.join(_fallback, "samples_final.csv")):
        DEPOSIT_DIR = _fallback
        print(f"[notebook] using sibling deposit layout: {DEPOSIT_DIR}")
REFERENCES_DIR = _os.environ.get("G_ROS_REFERENCES_DIR", "../references")   # see references/README.md

deposit = gr.load_deposit(DEPOSIT_DIR, REFERENCES_DIR, load_references=True)
print(f"Loaded {len(deposit.samples)} components × {deposit.samples.shape[1]} samples")


Loaded 547 components × 276 samples


## Configuration

In [3]:
# Choose which components to run. Defaults to the nine components shown
# in the manuscript figures. To sweep all 547 components, replace with:
#     COMPONENTS = list(deposit.samples.index)   # IDs 1–519 and 521–548
# To run a single component, e.g. just Component 74:
#     COMPONENTS = [74]
#
# Components and their manuscript figure (also in gr.HEADLINE_COMPONENTS):
#    74: Figure 2 (sex-associated regulatory variation)
#   278: Figure 3 (sex-biased component shared by OPCs and oligodendrocytes)
#   172: Figure 4 (neuron-dominated; neuritic plaque burden)
#   277: Figure 5 (neuron-dominated; clinical Alzheimer's dementia)
#    21: Sup Fig S2 (olfactory receptor expression and its cis-genetic regulation)
#   351: Sup Fig S8 (single-individual multi-omic profile)
#     2: Sup Fig S9 (single-individual multi-omic profile)
#    87: Sup Fig S10 (single-individual multi-omic profile)
#   177: Sup Fig S11 (dominated by one donor)

COMPONENTS = [74, 278, 172, 277, 21, 351, 2, 87, 177]

# Path to ROSMAP clinical CSV (controlled access — see DUC link), or set G_ROS_CLINICAL_PATH.
CLINICAL_PATH = os.environ.get('G_ROS_CLINICAL_PATH', os.path.join(REFERENCES_DIR, 'ROSMAP_clinical.csv'))

# Tests to run. Each requires its named column in the clinical CSV.
TESTS = ['sex', 'ceradsc', 'cogdx']

# Where to save the results table. None = display only.
SAVE_PATH = None  # e.g. "./out/clinical_associations.csv"


## Load clinical metadata

In [4]:
if not os.path.exists(CLINICAL_PATH):
    raise FileNotFoundError(
        f'ROSMAP_clinical.csv not found at {CLINICAL_PATH}.\n'
        'This file is controlled-access (request via the AD Knowledge Portal DUC).\n'
        'See https://adknowledgeportal.synapse.org/Data%20Access'
    )
clinical_df = pd.read_csv(CLINICAL_PATH, index_col=0)
print(f'Loaded clinical metadata: {clinical_df.shape}')
missing = [c for c in ['msex', 'ceradsc', 'cogdx'] if c not in clinical_df.columns]
if missing:
    print(f'WARNING — missing expected columns: {missing}')


Loaded clinical metadata: (3584, 17)


## Run association tests

In [5]:
all_results = gr.clinical_association_table(
    deposit,
    clinical_df,
    tests=TESTS,
)
results = all_results[all_results['component_id'].isin(COMPONENTS)].copy()

print(
    f'\nAdjusted across {deposit.samples.shape[0]} components per test; '
    f'displaying {len(COMPONENTS)} selected component(s).'
)
if not results.empty:
    display(results.sort_values(['test', 'p_adjusted', 'component_id']))



Adjusted across 547 components per test; displaying 9 selected component(s).


,component_id,test,n,statistic,p_value,n_male,n_female,n_ad,n_no,n_groups,p_adjusted
514,172,ceradsc,276,6726.000000,3.992492e-04,NaN,NaN,170.0,106.0,NaN,1.679918e-02
832,278,ceradsc,276,10574.000000,1.534150e-02,NaN,NaN,170.0,106.0,NaN,1.287864e-01
829,277,ceradsc,276,7638.000000,3.346011e-02,NaN,NaN,170.0,106.0,NaN,1.906529e-01
61,21,ceradsc,276,8095.000000,1.562090e-01,NaN,NaN,170.0,106.0,NaN,4.939095e-01
529,177,ceradsc,276,9740.000000,2.580156e-01,NaN,NaN,170.0,106.0,NaN,6.386178e-01
4,2,ceradsc,276,9477.000000,4.694889e-01,NaN,NaN,170.0,106.0,NaN,7.887776e-01
259,87,ceradsc,276,8681.000000,6.105132e-01,NaN,NaN,170.0,106.0,NaN,8.497473e-01
220,74,ceradsc,276,9260.000000,6.988664e-01,NaN,NaN,170.0,106.0,NaN,9.030850e-01
1051,351,ceradsc,276,8799.000000,7.441358e-01,NaN,NaN,170.0,106.0,NaN,9.167946e-01
830,277,cogdx,254,21.884257,1.769677e-05,NaN,NaN,NaN,NaN,3.0,2.420033e-03


## Significance summary

Data S1 uses Benjamini-Hochberg FDR correction within each clinical variable across all components. The summary below uses `p_adjusted < 0.05`.

In [6]:
if not all_results.empty:
    sig_all = all_results[all_results['p_adjusted'] < 0.05].sort_values(['test', 'p_adjusted'])
    sig_selected = sig_all[sig_all['component_id'].isin(COMPONENTS)]
    print(f'{len(sig_all)} significant component/test pair(s) across all components.')
    if sig_selected.empty:
        print('No selected components cross FDR < 0.05.')
    else:
        print(f'{len(sig_selected)} selected component/test pair(s) cross FDR < 0.05:')
        display(sig_selected)


73 significant component/test pair(s) across all components.
4 selected component/test pair(s) cross FDR < 0.05:


,component_id,test,n,statistic,p_value,n_male,n_female,n_ad,n_no,n_groups,p_adjusted
514,172,ceradsc,276,6726.000000,3.992492e-04,NaN,NaN,170.0,106.0,NaN,1.679918e-02
830,277,cogdx,254,21.884257,1.769677e-05,NaN,NaN,NaN,NaN,3.0,2.420033e-03
219,74,sex,276,18962.000000,1.448910e-46,129.0,147.0,NaN,NaN,NaN,7.925536e-44
831,278,sex,276,6808.000000,5.341981e-05,129.0,147.0,NaN,NaN,NaN,9.740212e-03


## Cross-check against Data S1

The `Sex`, `ceradsc`, and `cogdx` columns of `Data_S1.csv` are the FDR-adjusted p-values from these same tests; non-significant component/test pairs are left blank.

In [7]:
data_s1_path = '../Data_S1.csv'
if os.path.exists(data_s1_path):
    clinical_cols = ['Sex', 'ceradsc', 'cogdx']
    t1 = pd.read_csv(data_s1_path)
    published = t1[t1['Component'].isin(COMPONENTS)][['Component', *clinical_cols]].copy()
    published = published.rename(columns={c: f'{c}_Data_S1' for c in clinical_cols})

    local = results.copy()
    local['data_s1_style'] = local['p_adjusted'].where(local['p_adjusted'] < 0.05)
    local = local.pivot(index='component_id', columns='test', values='data_s1_style')
    local = local.rename(columns={'sex': 'Sex'}).reset_index()
    local = local.rename(columns={'component_id': 'Component'})
    for col in clinical_cols:
        if col not in local.columns:
            local[col] = np.nan
    local = local[['Component', *clinical_cols]].rename(columns={c: f'{c}_recomputed' for c in clinical_cols})

    comparison = published.merge(local, on='Component', how='left')
    for col in clinical_cols:
        comparison[f'{col}_match'] = np.isclose(
            comparison[f'{col}_Data_S1'].fillna(-1),
            comparison[f'{col}_recomputed'].fillna(-1),
            rtol=1e-10,
            atol=0,
        )
    display(comparison)
else:
    print('Data_S1.csv not found at the expected path; skipping cross-check.')


,Component,Sex_Data_S1,ceradsc_Data_S1,cogdx_Data_S1,Sex_recomputed,ceradsc_recomputed,cogdx_recomputed,Sex_match,ceradsc_match,cogdx_match
0,2,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True
1,21,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True
2,74,7.925536e-44,NaN,NaN,7.925536e-44,NaN,NaN,True,True,True
3,87,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True
4,172,NaN,0.016799,NaN,NaN,0.016799,NaN,True,True,True
5,177,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True
6,277,NaN,NaN,0.00242,NaN,NaN,0.00242,True,True,True
7,278,9.740212e-03,NaN,NaN,9.740212e-03,NaN,NaN,True,True,True
8,351,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True


## Save results

In [8]:
if SAVE_PATH is not None and not results.empty:
    results.to_csv(SAVE_PATH, index=False)
    print(f'Saved {len(results)} rows → {SAVE_PATH}')
